In [26]:
#cell 0
from pathlib import Path
import os
import sys

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)   # ✅ đổi về root để path tương đối đúng
print("CWD =", os.getcwd())


CWD = D:\logistics_AI


In [27]:
#cell 1
from __future__ import annotations
from pathlib import Path
import sys
import pandas as pd
import numpy as np

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from api.process_ai.inference import load_process_artifacts, analyze_batch


In [28]:
# Cell 2
import pandas as pd
from api.process_ai.inference import load_process_artifacts, analyze_batch

DATA_PATH = "data/synth_optimal_3process_v1/events.csv"
MODEL_DIR = "model/process_models"
REGISTRY_DIR = "registry/synth_optimal_v1"

PROCESS_CODES = [
    "TRUCKING_DELIVERY_FLOW",
    "WAREHOUSE_FULFILLMENT",
    "IMPORT_CUSTOMS_CLEARANCE"
]

events = pd.read_csv(DATA_PATH)

all_summaries = []
all_results = []

for pc in PROCESS_CODES:
    print("Scoring:", pc)

    events_pc = events[events["process_code"] == pc].copy()
    art = load_process_artifacts(pc, MODEL_DIR, REGISTRY_DIR)
    out = analyze_batch(events_pc, art, max_cases=None)

    summary = out["summary"]
    results = pd.DataFrame(out["results"]).copy()
    results["process_code"] = pc

    all_summaries.append(summary)
    all_results.append(results)

summary_df = pd.DataFrame(all_summaries)
results_df = pd.concat(all_results, ignore_index=True)

summary_df, len(results_df)

Scoring: TRUCKING_DELIVERY_FLOW
Scoring: WAREHOUSE_FULFILLMENT
Scoring: IMPORT_CUSTOMS_CLEARANCE


(               process_code  cases_analyzed  cases_ok   avg_risk  p95_risk
 0    TRUCKING_DELIVERY_FLOW           12000     12000  49.500083     94.05
 1     WAREHOUSE_FULFILLMENT           12000     12000  49.500083     94.05
 2  IMPORT_CUSTOMS_CLEARANCE            4286      4286  49.500700     94.75,
 28286)

In [29]:
# Cell 3
df = results_df.copy()

# nếu cột top_steps tồn tại và là list/dict thì bóc top step ra
def extract_top_step(x):
    if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict):
        return x[0]
    return {}

if "top_steps" in df.columns:
    top_info = df["top_steps"].apply(extract_top_step)
    df["top_step"] = top_info.apply(lambda x: x.get("step_code"))
    df["top_step_deviation"] = top_info.apply(lambda x: x.get("deviation_factor"))
    df["top_step_duration_min"] = top_info.apply(lambda x: x.get("duration_min"))
else:
    df["top_step"] = None
    df["top_step_deviation"] = None
    df["top_step_duration_min"] = None

# chọn cột cần xem
cols = [c for c in [
    "process_code",
    "case_id",
    "risk_score",
    "overall_severity",
    "top_step",
    "top_step_deviation",
    "top_step_duration_min"
] if c in df.columns]

df = df[cols].sort_values(["risk_score"], ascending=False)
df.head(20)

,process_code,case_id,risk_score,overall_severity,top_step,top_step_deviation,top_step_duration_min
7899,TRUCKING_DELIVERY_FLOW,ORD_2250406550,100.0,CRITICAL,STEP_019_IN_TRANSIT_LINEHAUL,3.048,1107.367
23289,WAREHOUSE_FULFILLMENT,ORD_0685870280,100.0,CRITICAL,STEP_003_PAYMENT_VERIFIED,2.316,1035.050
26648,IMPORT_CUSTOMS_CLEARANCE,ORD_4399774767,100.0,CRITICAL,STEP_002_DOC_VALIDATION,1.981,59.717
6710,TRUCKING_DELIVERY_FLOW,ORD_3693412107,99.0,CRITICAL,STEP_018_LINEHAUL_START,2.829,992.433
1564,TRUCKING_DELIVERY_FLOW,ORD_3050576017,99.0,CRITICAL,STEP_019_IN_TRANSIT_LINEHAUL,2.150,781.117
24733,IMPORT_CUSTOMS_CLEARANCE,ORD_6579634373,99.0,CRITICAL,STEP_009_DUTY_TAX_INVOICE_ISSUED,2.090,945.050
7993,TRUCKING_DELIVERY_FLOW,ORD_4549916655,99.0,CRITICAL,STEP_018_LINEHAUL_START,2.859,1002.733
22698,WAREHOUSE_FULFILLMENT,ORD_2543455046,99.0,CRITICAL,STEP_007_PICK_COMPLETE,2.223,55.717
8,TRUCKING_DELIVERY_FLOW,ORD_8397641323,99.0,CRITICAL,STEP_013_EN_ROUTE_HUB,2.709,930.467
18269,WAREHOUSE_FULFILLMENT,ORD_9932813332,99.0,CRITICAL,STEP_003_PAYMENT_VERIFIED,2.638,1178.883


In [30]:
#cell 4
# Lấy start_time đầu tiên mỗi case để group theo ngày
tmp = events[events["process_code"] == PROCESS_CODE].copy()
tmp["start_time"] = pd.to_datetime(tmp["start_time"], errors="coerce")
case_start = tmp.groupby("case_id")["start_time"].min().rename("case_start_time").reset_index()

df2 = df.merge(case_start, on="case_id", how="left")
df2["date"] = df2["case_start_time"].dt.date

daily = df2.groupby("date").agg(
    cases=("case_id","count"),
    avg_risk=("risk_score","mean"),
    p95_risk=("risk_score", lambda x: np.quantile(x, 0.95)),
    bottleneck_rate=("overall_severity", lambda x: (x.isin(["BOTTLENECK","CRITICAL"])).mean())
).reset_index()

daily.head()


,date,cases,avg_risk,p95_risk,bottleneck_rate
0,2025-04-01,60,48.166667,93.05,0.450000
1,2025-04-02,158,50.120253,94.00,0.537975
2,2025-04-03,223,47.753363,94.00,0.493274
3,2025-04-04,299,49.083612,95.00,0.484950
4,2025-04-05,286,50.486014,94.75,0.520979


In [31]:
#cell 5
Path("test").mkdir(exist_ok=True)
df2.to_csv(f"test/process_report_{PROCESS_CODE}.csv", index=False, encoding="utf-8-sig")
daily.to_csv(f"test/process_daily_{PROCESS_CODE}.csv", index=False, encoding="utf-8-sig")
print("Saved:", f"test/process_report_{PROCESS_CODE}.csv", "and", f"test/process_daily_{PROCESS_CODE}.csv")


Saved: test/process_report_TRUCKING_DELIVERY_FLOW.csv and test/process_daily_TRUCKING_DELIVERY_FLOW.csv
